# AkinaCopa

---

In [2]:
import pandas as pd
import numpy as np

## Bloco 1: Matriz de Candidatos

Le os dois CSVs, cruza os dados e cria uma coluna binaria para cada atributo.

In [ ]:
COPAS_MODERNAS = [2002, 2006, 2010, 2014, 2018, 2022]

def construir_matriz_participacoes(caminho_jogadores, caminho_participacoes,
                                   copas_ativas=None):
    """
    Constrói a matriz de candidatos.
    copas_ativas: lista de anos para filtrar o escopo (ex: COPAS_MODERNAS).
                  None = todas as Copas.
    """
    df_jogadores     = pd.read_csv(caminho_jogadores)
    df_participacoes = pd.read_csv(caminho_participacoes)
    df = pd.merge(df_participacoes, df_jogadores, on="id_jogador")

    # Filtro de escopo
    if copas_ativas:
        df = df[df['ano_copa'].isin(copas_ativas)].copy()

    # Linha de campo
    for pos in ['Atacante', 'Goleiro', 'Defensor', 'Meio-campista']:
        df[pos] = (df['linha_campo'] == pos).astype(int)

    # Posição específica
    for pos in ['Zagueiro', 'Lateral', 'Ponta', 'Volante', 'Meia-atacante', 'Centroavante']:
        df[pos] = (df['posicao_especifica'] == pos).astype(int)

    # Épocas
    df['antes_1970'] = (df['ano_copa'] < 1970).astype(int)
    df['anos_70_80'] = ((df['ano_copa'] >= 1974) & (df['ano_copa'] <= 1990)).astype(int)
    df['seculo_21']  = (df['ano_copa'] >= 2002).astype(int)
    df['era_pele']   = df['ano_copa'].isin([1958, 1962, 1966, 1970]).astype(int)
    for ano in [1994, 2002, 2006, 2010, 2014, 2018, 2022]:
        df[f'copa_{ano}'] = (df['ano_copa'] == ano).astype(int)

    # Conquistas
    df['campeao']        = df['ano_copa'].isin([1958, 1962, 1970, 1994, 2002]).astype(int)
    df['capitao']        = (df['capitao'] == 'Sim').astype(int)
    # Veterano calculado dentro do escopo ativo
    contagem_copas       = df.groupby('id_jogador')['ano_copa'].nunique()
    df['veterano']       = df['id_jogador'].map(contagem_copas).gt(1).astype(int)
    df['super_veterano'] = df['id_jogador'].map(contagem_copas).ge(3).astype(int)
    df['finalista']       = 0
    df['campeao_invicto'] = 0

    # Clubes e Regiões
    clubes_rj = ['Flamengo', 'Vasco da Gama', 'Botafogo', 'Fluminense']
    clubes_sp = ['São Paulo', 'Palmeiras', 'Corinthians', 'Santos', 'Portuguesa']
    df['RJ']      = df['clube_no_ano'].str.contains('|'.join(clubes_rj), na=False).astype(int)
    df['SP']      = df['clube_no_ano'].str.contains('|'.join(clubes_sp), na=False).astype(int)
    df['exterior'] = (df['clube_pais'] != 'Brasil').astype(int)

    for clube in ['Flamengo', 'Vasco da Gama', 'Botafogo', 'Fluminense',
                  'São Paulo', 'Palmeiras', 'Corinthians', 'Santos']:
        df[clube] = df['clube_no_ano'].str.contains(clube, na=False).astype(int)

    df['Cruzeiro']         = df['clube_no_ano'].str.contains('Cruzeiro', na=False).astype(int)
    df['Atlético Mineiro'] = df['clube_no_ano'].str.contains('Atlético Mineiro', na=False).astype(int)
    df['Grêmio']           = df['clube_no_ano'].str.contains('Grêmio', na=False).astype(int)
    df['Internacional']    = df['clube_no_ano'].str.contains(r'\bInternacional\b', na=False).astype(int)

    # Países adicionais relevantes para era moderna
    df['pais_Franca']    = (df['clube_pais'] == 'França').astype(int)
    df['pais_Alemanha']  = (df['clube_pais'] == 'Alemanha').astype(int)
    df['pais_Portugal']  = (df['clube_pais'] == 'Portugal').astype(int)

    # Atributos por participação
    df['titular']         = (df['titular'] == 'Sim').astype(int)
    df['gols_positivo']   = (df['gols_na_copa'] > 0).astype(int)
    df['gols_artilheiro'] = (df['gols_na_copa'] >= 3).astype(int)
    df['lado_esquerdo']   = (df['lado_campo'] == 'Esquerdo').astype(int)
    df['lado_direito']    = (df['lado_campo'] == 'Direito').astype(int)
    df['pais_Inglaterra'] = (df['clube_pais'] == 'Inglaterra').astype(int)
    df['pais_Espanha']    = (df['clube_pais'] == 'Espanha').astype(int)
    df['pais_Italia']     = (df['clube_pais'] == 'Itália').astype(int)
    df['expulso']         = (df['expulso_ou_suspenso'] == 'Sim').astype(int)
    df['artilheiro']      = (df['artilheiro_destaque'] == 'Sim').astype(int)
    df['sub23']           = (df['faixa_etaria'] == 'Sub-23').astype(int)
    df['altura_alto']     = (df['altura_categoria'] == 'Alto').astype(int)

    df['total_erros'] = 0.0
    return df


def filtrar_perguntas_para_escopo(df_perguntas, matriz):
    """Remove perguntas sem variância na matriz atual (coluna toda 0 ou toda 1)."""
    cols = set(matriz.columns)
    perguntas_validas = []
    for _, row in df_perguntas.iterrows():
        chave = row['chave_atributo']
        partes = [p.strip() for p in chave.split(',')]
        validas = [p for p in partes if p in cols]
        if not validas:
            continue
        # Verifica se ao menos uma parte tem variância
        tem_variancia = any(
            0 < matriz[p].mean() < 1.0 for p in validas
        )
        if tem_variancia:
            perguntas_validas.append(row)
    return pd.DataFrame(perguntas_validas).reset_index(drop=True)

---
## Bloco 2: Mapa de Respostas

Converte a escolha do usuario (1 a 5) em um peso numerico entre 0 e 1.

| Resposta | Peso |
|---|---|
| Sim | 1.0 |
| Provavelmente Sim | 0.75 |
| Não Sei | (sem penalidade) |
| Provavelmente Não | 0.25 |
| Não | 0.0 |

In [4]:
MAPA_RESPOSTAS = {
    "1": 1.0,    # Sim
    "2": 0.75,   # Provavelmente Sim
    "3": None,   # Não Sei (sem penalidade)
    "4": 0.25,   # Provavelmente Não
    "5": 0.0,    # Não
}

---
## Bloco 3: Resolucão de Chave Composta

Algumas perguntas cobrem mais de um atributo ao mesmo tempo.

Exemplo: `chave_atributo = "Lateral, Ponta"` significa "é lateral OU ponta?".  
A função faz um OR binário entre as colunas correspondentes na matriz.

In [5]:
def resolver_chave(chave, df, colunas_disponiveis):
    """
    Retorna uma Série booleana (0/1) para chaves simples OU compostas (OR entre colunas).
    Ex: "Lateral, Ponta" → df['Lateral'] | df['Ponta']
    """
    partes = [p.strip() for p in chave.split(',')]
    partes_validas = [p for p in partes if p in colunas_disponiveis]
    if not partes_validas:
        return None
    serie = df[partes_validas[0]].copy()
    for p in partes_validas[1:]:
        serie = serie | df[p]
    return serie.astype(int)

---
## Bloco 4: Exclusões Contextuais

Após uma resposta, certas perguntas se tornam redundantes.

Exemplos:
- Confirmado **Goleiro** → remove perguntas sobre gols, lado de campo e posicoes de linha
- Confirmado **copa_2022** → remove todas as outras copas-ano
- Confirmado **exterior** → remove todos os clubes brasileiros

Isso reduz drasticamente o número de perguntas necessárias para um palpite preciso.

In [6]:
_COPAS_ANO  = ['copa_2022', 'copa_2018', 'copa_2014', 'copa_2010', 'copa_2006', 'copa_2002', 'copa_1994']
_ERA_ANTIGA = ['antes_1970', 'anos_70_80', 'era_pele']

_CLUBES_BR = [
    'RJ', 'SP',
    'Flamengo', 'Botafogo', 'Fluminense',
    'São Paulo', 'Palmeiras', 'Corinthians', 'Santos',
    'Cruzeiro', 'Atlético Mineiro', 'Grêmio', 'Internacional',
]
_PAISES = [
    'pais_Inglaterra', 'pais_Espanha', 'pais_Italia',
    'pais_Franca', 'pais_Alemanha', 'pais_Portugal',
]

EXCLUSOES_CONTEXTUAIS = {

    # Linha de campo
    'Atacante':      {'excluir_se_sim': ['Goleiro', 'Defensor', 'Meio-campista', 'Zagueiro',
                                         'Lateral', 'Volante', 'Meia-atacante']},
    'Goleiro':       {'excluir_se_sim': ['Atacante', 'Defensor', 'Meio-campista', 'Zagueiro',
                                         'Lateral', 'Ponta', 'Centroavante', 'Volante', 'Meia-atacante',
                                         'lado_esquerdo', 'lado_direito',
                                         'gols_positivo', 'gols_artilheiro', 'artilheiro']},
    'Defensor':      {'excluir_se_sim': ['Atacante', 'Goleiro', 'Meio-campista',
                                         'Ponta', 'Centroavante', 'Meia-atacante']},
    'Meio-campista': {'excluir_se_sim': ['Atacante', 'Goleiro', 'Defensor',
                                         'Zagueiro', 'Lateral', 'Ponta', 'Centroavante']},

    # Posições específicas
    'Zagueiro':      {'excluir_se_sim': ['Lateral', 'Ponta', 'Centroavante', 'Volante',
                                         'Meia-atacante', 'Goleiro',
                                         'lado_esquerdo', 'lado_direito']},
    'Lateral':       {'excluir_se_sim': ['Zagueiro', 'Ponta', 'Centroavante', 'Goleiro', 'Atacante']},
    'Ponta':         {'excluir_se_sim': ['Zagueiro', 'Lateral', 'Centroavante', 'Volante', 'Goleiro',
                                         'Atacante']},       # Ponta é subtype de Atacante → trivial
    'Centroavante':  {'excluir_se_sim': ['Zagueiro', 'Lateral', 'Ponta', 'Volante',
                                         'Meia-atacante', 'Goleiro',
                                         'lado_esquerdo', 'lado_direito',
                                         'Atacante']},       # CA é subtype de Atacante → trivial
    'Volante':       {'excluir_se_sim': ['Zagueiro', 'Lateral', 'Ponta', 'Centroavante',
                                         'Goleiro', 'lado_esquerdo', 'lado_direito']},
    'Meia-atacante': {'excluir_se_sim': ['Zagueiro', 'Lateral', 'Centroavante', 'Volante',
                                         'Goleiro', 'lado_esquerdo', 'lado_direito']},

    # Lado do campo
    'lado_esquerdo': {'excluir_se_sim': ['lado_direito']},
    'lado_direito':  {'excluir_se_sim': ['lado_esquerdo']},

    # Épocas
    'seculo_21':  {
        'excluir_se_sim': _ERA_ANTIGA,
        'excluir_se_nao': _COPAS_ANO + _PAISES,
    },
    'antes_1970': {'excluir_se_nao': ['era_pele']},
    'era_pele':   {'excluir_se_nao': ['antes_1970']},

    # Copa-ano
    **{
        f'copa_{ano}': {
            'excluir_se_sim': [c for c in _COPAS_ANO if c != f'copa_{ano}']
                              + _ERA_ANTIGA + ['seculo_21']
        }
        for ano in [2022, 2018, 2014, 2010, 2006, 2002, 1994]
    },

    # Conquistas
    'campeao': {'excluir_se_nao': ['campeao_invicto']},
    'capitao': {'excluir_se_sim': ['titular']},          # capitão sempre foi titular

    # Gols
    'gols_positivo':   {'excluir_se_nao': ['gols_artilheiro']},   # 0 gols → não pode ter 3+
    'gols_artilheiro': {'excluir_se_sim': ['gols_positivo']},      # 3+ → 1+ trivial
    'artilheiro':      {'excluir_se_sim': ['gols_positivo',        # artilheiro → tem gols
                                            'gols_artilheiro']},

    # Veterano
    'veterano':       {'excluir_se_nao': ['super_veterano']},     # 1 Copa → não pode ser 3+
    'super_veterano': {'excluir_se_sim': ['veterano']},            # 3+ → veterano trivial

    # exterior=Sim TODOS os clubes brasileiros são impossíveis
    'exterior': {
        'excluir_se_sim': _CLUBES_BR,
        'excluir_se_nao': _PAISES,
    },

    # País confirmado -> exterior trivial + outros países impossíveis + clubes BR impossíveis
    **{
        pais: {
            'excluir_se_sim': ['exterior']
                              + [p for p in _PAISES if p != pais]
                              + _CLUBES_BR
        }
        for pais in _PAISES
    },

    # Clube BR confirmado -> exterior impossível + países impossíveis
    **{
        clube: {'excluir_se_sim': ['exterior'] + _PAISES}
        for clube in ['Flamengo', 'Botafogo', 'Fluminense',
                      'São Paulo', 'Palmeiras', 'Corinthians', 'Santos',
                      'Cruzeiro', 'Atlético Mineiro', 'Grêmio', 'Internacional']
    },

    # Estado confirmado → exterior/países impossíveis + estado oposto + clubes do estado oposto
    'RJ': {
        'excluir_se_sim': ['exterior'] + _PAISES
                          + ['SP', 'São Paulo', 'Palmeiras', 'Corinthians', 'Santos',
                             'Cruzeiro', 'Atlético Mineiro', 'Grêmio', 'Internacional'],
    },
    'SP': {
        'excluir_se_sim': ['exterior'] + _PAISES
                          + ['RJ', 'Flamengo', 'Botafogo', 'Fluminense',
                             'Cruzeiro', 'Atlético Mineiro', 'Grêmio', 'Internacional'],
    },
}


def aplicar_exclusoes(perguntas_restantes, chave_respondida, peso_usuario):
    """Remove da fila perguntas redundantes pelo contexto da resposta."""
    regras = EXCLUSOES_CONTEXTUAIS.get(chave_respondida, {})
    chaves_para_excluir = set()

    if peso_usuario >= 0.75:
        chaves_para_excluir.update(regras.get('excluir_se_sim', []))
    elif peso_usuario <= 0.25:
        chaves_para_excluir.update(regras.get('excluir_se_nao', []))

    if not chaves_para_excluir:
        return perguntas_restantes

    def deve_excluir(chave_pergunta):
        partes = {p.strip() for p in chave_pergunta.split(',')}
        return bool(partes & chaves_para_excluir)

    return perguntas_restantes[~perguntas_restantes['chave_atributo'].apply(deve_excluir)]

---
## Bloco 5: Seleção da Melhor Pergunta

A pergunta ideal é aquela que divide os candidatos restantes o mais proximo de **50% / 50%**.

Isso maximiza a informacao obtida a cada rodada.

**Estratégia dual:**
- Poucos candidatos no topo (5 ou menos): modo preciso, foca nos finalistas
- Muitos candidatos: modo amplo, usa o quartil inferior como filtro

In [7]:
def calcular_melhor_pergunta(df_jogo, perguntas_disponiveis, colunas_disponiveis):
    """
    Escolhe a pergunta que mais discrimina os candidatos ativos.

    Estratégia dual:
    - Grupo pequeno (≤5 candidatos no topo): foca só nesses para discriminação precisa.
    - Grupo grande: usa quartil inferior para filtro amplo.
    """
    if len(df_jogo) <= 1:
        return None

    min_erro        = df_jogo['total_erros'].min()
    candidatos_top  = df_jogo[df_jogo['total_erros'] <= min_erro + 0.5]

    if len(candidatos_top) <= 5:
        # Modo preciso: encontra a pergunta que separa os poucos finalistas
        candidatos_ativos = candidatos_top
        if len(candidatos_ativos) <= 1:
            candidatos_ativos = df_jogo
    else:
        # Modo amplo: quartil inferior do grupo completo
        limiar = df_jogo['total_erros'].quantile(0.25)
        candidatos_ativos = df_jogo[df_jogo['total_erros'] <= limiar + 0.001]
        if len(candidatos_ativos) <= 1:
            candidatos_ativos = df_jogo

    melhor_pergunta = None
    menor_distancia = float('inf')

    for _, row in perguntas_disponiveis.iterrows():
        chave = row['chave_atributo']
        serie = resolver_chave(chave, candidatos_ativos, colunas_disponiveis)
        if serie is None:
            continue
        proporcao_sim = serie.mean()
        distancia = abs(proporcao_sim - 0.5)
        if distancia < menor_distancia:
            menor_distancia = distancia
            melhor_pergunta = row

    return melhor_pergunta

---
## Bloco 6: Motor do Jogo

Loop principal que une todos os blocos anteriores:

1. Verifica se ja há confianca para um palpite intermediário (margem >= 1.5 após rodada 4)
2. Seleciona a melhor pergunta disponivel (Bloco 5)
3. Aplica a resposta — calcula erros (Bloco 2) e remove perguntas redundantes (Bloco 4)
4. Repete por até 22 rodadas
5. Ao final, faz o palpite definitivo com o candidato de menor erro acumulado

In [8]:
def jogar_akinator(matriz, df_perguntas):
    df_jogo = matriz.copy()
    df_jogo['total_erros'] = 0.0

    colunas_disponiveis = set(df_jogo.columns)
    perguntas_restantes = df_perguntas.copy()
    rodada = 1
    MAX_RODADAS = 22

    print("Pense em um jogador da Seleção Brasileira em uma Copa específica\n")

    while rodada <= MAX_RODADAS and not perguntas_restantes.empty:

        # Verifica se ja ha confianca para um palpite intermediario
        df_ord = df_jogo.sort_values('total_erros')
        erro_1o = df_ord.iloc[0]['total_erros']
        erro_2o = df_ord.iloc[1]['total_erros'] if len(df_ord) > 1 else float('inf')
        margem  = erro_2o - erro_1o

        if rodada >= 4 and margem >= 1.5:
            candidato = df_ord.iloc[0]
            nome_copa = f"{candidato['nome']} na Copa de {int(candidato['ano_copa'])}"
            print(f"\nEstou pensando em: {nome_copa}")
            conf = input("  (S = sim / N = não / C = continuar): ").strip().upper()
            if conf == 'S':
                print(f"\nAcertei! O jogador era {nome_copa}.")
                return df_jogo
            elif conf == 'N':
                df_jogo.loc[df_ord.index[0], 'total_erros'] += 5.0
                print("  Ok, vou continuar...\n")
            # 'C' segue sem palpitar

        pergunta = calcular_melhor_pergunta(df_jogo, perguntas_restantes, colunas_disponiveis)
        if pergunta is None:
            break

        chave = pergunta['chave_atributo']
        perguntas_restantes = perguntas_restantes[
            perguntas_restantes['id_pergunta'] != pergunta['id_pergunta']
        ]

        serie = resolver_chave(chave, df_jogo, colunas_disponiveis)
        if serie is None:
            continue

        print(f"Pergunta {rodada}: {pergunta['texto_pergunta']}")
        print("1) Sim  2) Provavelmente Sim  3) Não Sei  4) Provavelmente Não  5) Não")

        while True:
            resp = input("Sua resposta (1-5): ").strip()
            if resp in MAPA_RESPOSTAS:
                break
            print("Resposta inválida. Digite um numero de 1 a 5.")

        peso_usuario = MAPA_RESPOSTAS[resp]

        if peso_usuario is not None:
            df_jogo['total_erros'] += abs(serie - peso_usuario)
            perguntas_restantes = aplicar_exclusoes(perguntas_restantes, chave, peso_usuario)

        print()
        rodada += 1

    # Palpite final
    df_jogo   = df_jogo.sort_values('total_erros')
    palpite   = df_jogo.iloc[0]
    nome_copa = f"{palpite['nome']} na Copa de {int(palpite['ano_copa'])}"

    print("=" * 50)
    print(f"PALPITE FINAL: {nome_copa}")
    print(f"Fato marcante: {palpite['caracteristica_geral']}")
    print(f"Erro acumulado: {palpite['total_erros']:.2f}")
    print("=" * 50)

    print("\nTop 3 candidatos internos:")
    for i, (_, row) in enumerate(df_jogo.head(3).iterrows(), 1):
        print(f"  {i}. {row['nome']} ({int(row['ano_copa'])}) — erro: {row['total_erros']:.2f}")

    feedback = input("\nAcertei? (S/N): ").strip().upper()
    if feedback == 'S':
        print("\nO motor acertou.")
    else:
        nome_real = input("Quem era? (ex: Ronaldo 2002): ").strip()
        print(f"\nRegistrado! {nome_real} escapou desta vez.")

    return df_jogo

---
## Executando o Motor

In [ ]:
matriz = construir_matriz_participacoes(
    "data/akinacopa-dataset - Jogadores.csv",
    "data/akinacopa-dataset - Participações Copa.csv",
    copas_ativas=COPAS_MODERNAS,
)
perguntas = pd.read_csv("data/akinacopa-dataset - Perguntas.csv")
perguntas = filtrar_perguntas_para_escopo(perguntas, matriz)

print(f"Base carregada: {len(matriz)} participacoes | {len(perguntas)} perguntas ativas")
print("-" * 50)
jogar_akinator(matriz, perguntas)